# CroSloEngual BERT — Slovene Gender Classification

Fine-tunes `EMBEDDIA/crosloengual-bert` (Croatian + Slovenian + English) on `janes_blog_gender.csv` (Janes-Blog corpus, gender pre-labeled).
Evaluates on the Slovenian subset of LiLaH.

**Before running:**
1. Runtime → Change runtime type → **GPU** (T4)
2. Upload `janes_blog_gender.csv` to `My Drive/thesis/`
3. Upload `merged-sl-meta-lilah.tsv` to `My Drive/thesis/` (if not already there)
4. Run all cells top to bottom

Model saves to `My Drive/thesis/croslobert_gender_model/`

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Check GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — go to Runtime > Change runtime type > GPU')

In [ ]:
!pip install -q -U transformers accelerate
import transformers
print(f'transformers: {transformers.__version__}')

In [ ]:
# ── CONFIGURATION ─────────────────────────────────────────────────────────
# CroSloEngual BERT: trained on Croatian + Slovenian + English
# Uses standard BERT WordPiece tokenizer — no sentencepiece compatibility issues
MODEL_NAME  = 'EMBEDDIA/crosloengual-bert'

TRAIN_CSV   = '/content/drive/MyDrive/thesis/janes_blog_gender.csv'
LILAH_TSV   = '/content/drive/MyDrive/thesis/merged-sl-meta-lilah.tsv'
OUTPUT_DIR  = '/content/drive/MyDrive/thesis/croslobert_gender_model'

SAMPLE_PER_CLASS = 30000   # 30k M + 30k F = 60k total, balanced

MAX_LENGTH        = 256
TRAIN_BATCH_SIZE  = 8
EVAL_BATCH_SIZE   = 16
GRAD_ACCUM_STEPS  = 4      # effective batch = 32
NUM_EPOCHS        = 3
LEARNING_RATE     = 2e-5
WEIGHT_DECAY      = 0.01
RANDOM_STATE      = 42
LABELS            = ['F', 'M']
# ──────────────────────────────────────────────────────────────────────────
print(f'Model: {MODEL_NAME}')
print(f'Output: {OUTPUT_DIR}')

In [ ]:
import os, json, time
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, classification_report, precision_recall_fscore_support
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_length, return_tensors=None
        )
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

    def __len__(self):
        return len(self.labels)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = logits.argmax(axis=1)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average='macro', zero_division=0
    )
    return {
        'accuracy': accuracy_score(labels, preds),
        'macro_f1': f1,
        'macro_precision': precision,
        'macro_recall': recall,
    }

print('Imports OK')

In [ ]:
# Load Janes-Blog training data
df = pd.read_csv(TRAIN_CSV)
df = df.dropna(subset=['gender', 'text']).copy()
df['text'] = df['text'].astype(str)
df = df[df['gender'].isin(LABELS)].reset_index(drop=True)

print(f'Full corpus: {len(df)} texts')
print(df['gender'].value_counts())

# Balanced sample — cap per class to avoid memory pressure
sampled = (
    df.groupby('gender', group_keys=False)
      .apply(lambda g: g.sample(min(len(g), SAMPLE_PER_CLASS), random_state=RANDOM_STATE))
)
sampled = sampled.sample(frac=1, random_state=RANDOM_STATE).reset_index(drop=True)

print(f'\nTraining sample: {len(sampled)} texts')
print(sampled['gender'].value_counts())
print(f'Avg text length: {sampled["text"].str.len().mean():.0f} chars')

In [ ]:
# Label encoding + train/val split
label2id = {label: idx for idx, label in enumerate(LABELS)}
id2label  = {idx: label for label, idx in label2id.items()}
sampled['label'] = sampled['gender'].map(label2id)

train_df, val_df = train_test_split(
    sampled, test_size=0.1, random_state=RANDOM_STATE, stratify=sampled['label']
)
print(f'Train: {len(train_df)} | Val: {len(val_df)}')

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(os.path.join(OUTPUT_DIR, 'label_mapping.json'), 'w') as f:
    json.dump({'task': 'gender', 'label2id': label2id,
               'id2label': {str(k): v for k, v in id2label.items()}}, f, indent=2)
print('Label mapping saved')

In [ ]:
# Tokenize — standard BERT WordPiece, no use_fast issues
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizing training set...')
train_dataset = TextDataset(train_df['text'].tolist(), train_df['label'].tolist(), tokenizer, MAX_LENGTH)
print('Tokenizing validation set...')
val_dataset   = TextDataset(val_df['text'].tolist(),   val_df['label'].tolist(),   tokenizer, MAX_LENGTH)
print(f'Train: {len(train_dataset)} | Val: {len(val_dataset)}')

In [ ]:
# Load SloBERTa
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=len(LABELS), id2label=id2label, label2id=label2id
)
print(f'Parameters: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# Train
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    greater_is_better=True,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,
    dataloader_pin_memory=False,
    report_to='none',
    seed=RANDOM_STATE,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

start = time.time()
trainer.train()
print(f'Done in {round((time.time()-start)/60, 1)} min')

In [ ]:
# Save model
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to: {OUTPUT_DIR}')

In [ ]:
# Validation performance (in-domain — Janes-Blog)
metrics = trainer.evaluate()
print('\nValidation metrics (Janes-Blog):')
for k, v in metrics.items():
    print(f'  {k}: {round(v, 4) if isinstance(v, float) else v}')

preds_out  = trainer.predict(val_dataset)
pred_ids   = preds_out.predictions.argmax(axis=1)
true_ids   = preds_out.label_ids
print('\nClassification report (Janes-Blog val):')
print(classification_report(true_ids, pred_ids, target_names=LABELS, zero_division=0))

## Evaluate on LiLaH SL (cross-dataset)

Cross-dataset evaluation: trained on Janes-Blog (blogs/comments), tested on LiLaH (Facebook hate groups).  
This mirrors the EN setup: PAN14 → LiLaH EN.

In [ ]:
# Load LiLaH SL
lilah = pd.read_csv(LILAH_TSV, sep='\t')
lilah = lilah.dropna(subset=['text', 'gender']).copy()
lilah['text'] = lilah['text'].astype(str)

# Normalise gender labels
gender_norm = {'male': 'M', 'female': 'F', 'm': 'M', 'f': 'F', 'M': 'M', 'F': 'F'}
lilah['gender'] = lilah['gender'].astype(str).str.strip().map(
    lambda x: gender_norm.get(x, x.upper() if x.upper() in ['M', 'F'] else None)
)
lilah = lilah[lilah['gender'].isin(LABELS)].reset_index(drop=True)

print(f'LiLaH SL: {len(lilah)} texts')
print(lilah['gender'].value_counts())

In [ ]:
# Predict on LiLaH SL
eval_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.eval()
model.to(eval_device)

BATCH = 64
all_preds = []
texts_all = lilah['text'].tolist()

for i in range(0, len(texts_all), BATCH):
    batch = texts_all[i:i+BATCH]
    enc = tokenizer(batch, truncation=True, padding=True,
                    max_length=MAX_LENGTH, return_tensors='pt')
    enc = {k: v.to(eval_device) for k, v in enc.items()}
    with torch.no_grad():
        logits = model(**enc).logits
    all_preds.extend(torch.argmax(logits, dim=1).cpu().tolist())

pred_labels = [id2label[i] for i in all_preds]
true_labels = lilah['gender'].tolist()

print('=== LiLaH SL Cross-Dataset Results ===')
print(f'Macro F1: {f1_score(true_labels, pred_labels, average="macro", labels=LABELS, zero_division=0):.4f}')
print(classification_report(true_labels, pred_labels, labels=LABELS, zero_division=0))

# Save predictions
LILAH_OUTPUT = OUTPUT_DIR + '/sloberta_gender_lilah_predictions.csv'
out_df = lilah.copy()
out_df['predicted'] = pred_labels
out_df.to_csv(LILAH_OUTPUT, index=False)
print(f'Predictions saved to: {LILAH_OUTPUT}')